# RailwaySafety: mevcut modeli doğrula, gerekirse fine-tune et

Bu defter mevcut best.pt dosyasını değiştirmez. Önce sınıfları, veri bölünmesini ve test başarısını kontrol et. Gevşek bağlantı ayrı ve güvenilir etiket gerektirir. Colab kullanıyorsan repo ve veri klasörünü yükle/Drive üzerinden bağla; GPU çalışma zamanını seç.

In [ ]:
%pip install ultralytics==8.4.198 Pillow numpy PyYAML


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

# Bu iki yolu kendi ortamına göre düzenle.
ROOT = Path("/content/RailwaySafety") if Path("/content").exists() else Path.cwd()
if not (ROOT / "assets/best.pt").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data/data.yaml"
WEIGHTS = ROOT / "assets/best.pt"
assert DATA.is_file() and WEIGHTS.is_file(), "ROOT ve DATA yollarını düzenle"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(WEIGHTS))
print(model.task, model.names, DEVICE)


## Veri denetimi

Kaynak/güzergah bazlı ayrım, sağlam ray negatifleri ve sınıf dengesi görsel olarak da incelenmeli. Aşağıdaki kontrol sadece koordinatları, sayıları ve birebir kopyaları denetler.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from apps.audit_dataset import audit
audit_report = audit(DATA)
print(audit_report["splits"])
print("Invalid labels:", len(audit_report["issues"]))
print("Cross-split exact duplicates:", len(audit_report["cross_split_exact_duplicates"]))
assert not audit_report["issues"] and not audit_report["cross_split_exact_duplicates"]


In [ ]:
# Eşik/model seçimi için validation kullan; test bölümünü seçim döngüsüne katma.
baseline = model.val(data=str(DATA), split="val", device=DEVICE, workers=0, imgsz=640)
print(baseline.results_dict)


## İsteğe bağlı fine-tuning

Bu hücreyi yalnızca veri etiketleri/dengesi iyileştirildikten sonra çalıştır. Yeni ağırlık ayrı run altında oluşur. Aynı veriyle daha fazla epoch başarılı olacağını garanti etmez.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    model.train(data=str(DATA), epochs=50, imgsz=640, batch=8, device=DEVICE,
                workers=0, seed=0, project=str(ROOT / "runs/railway_training"),
                name="finetune", exist_ok=False)
    CANDIDATE = Path(model.trainer.best)
else:
    CANDIDATE = WEIGHTS
print("Candidate:", CANDIDATE)


In [ ]:
# Model seçimi tamamlandıktan sonra bağımsız test.
candidate = YOLO(str(CANDIDATE))
metrics = candidate.val(data=str(DATA), split="test", device=DEVICE, workers=0, imgsz=640)
print(metrics.results_dict)
for class_id in metrics.box.ap_class_index:
    print(candidate.names[int(class_id)], float(metrics.box.maps[int(class_id)]))


Yeni ağırlığı otomatik olarak assets/best.pt üzerine kopyalama. Sınıf bazlı recall, yanlış alarm ve ayrı saha videosu değerlendirmesi sonrası sürümlü olarak seç. Test sonucu işletme güvenliği sertifikası değildir.